# Stage 4 — Case Solution Reuse

**Objective:** Predict a solution for query cases using retrieved similar cases.

In [1]:
import json
import pandas as pd
from pathlib import Path
from collections import Counter

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
EVAL_DIR = PROJECT_ROOT / 'data' / 'eval'
RESULTS_DIR = PROJECT_ROOT / 'data' / 'results'

cases_df = pd.read_csv(PROCESSED_DIR / 'cases.csv')
with open(RESULTS_DIR / 'retrieval_baseline.json', 'r', encoding='utf-8') as f:
    retrieval_results = json.load(f)
with open(EVAL_DIR / 'queries.json', 'r', encoding='utf-8') as f:
    eval_queries = json.load(f)

import re
def get_primary_pasal(pasal_str):
    if pd.isna(pasal_str): return 'Lainnya'
    for part in pasal_str.split(';'):
        m = re.match(r'^\s*(\d+)', part.strip())
        if m:
            num = int(m.group(1))
            if 27 <= num <= 37: return f'Pasal {num}'
    return 'Lainnya'

cases_df['primary_pasal'] = cases_df['pasal'].apply(get_primary_pasal)
amar_lookup = dict(zip(cases_df['case_id'], cases_df['amar_putusan']))
pasal_lookup = dict(zip(cases_df['case_id'], cases_df['primary_pasal']))

In [2]:
def majority_vote_pasal(retrieved_cases):
    labels = [pasal_lookup.get(r['case_id'], 'Lainnya') for r in retrieved_cases]
    if not labels: return 'Lainnya'
    counter = Counter(labels)
    max_count = counter.most_common(1)[0][1]
    tied = [label for label, count in counter.items() if count == max_count]
    for r in retrieved_cases:
        label = pasal_lookup.get(r['case_id'], 'Lainnya')
        if label in tied: return label
    return tied[0]

def reuse_solution(retrieved_cases):
    if not retrieved_cases: return ''
    return amar_lookup.get(retrieved_cases[0]['case_id'], '')

predictions = []
for q in eval_queries:
    qid = q['query_id']
    res = retrieval_results[qid]
    
    cos_cases = res['cosine_results']
    svm_cases = res['svm_results']
    
    predictions.append({
        'query_id': qid,
        'actual_pasal': q['expected_pasal'],
        'cosine_predicted_pasal': majority_vote_pasal(cos_cases),
        'svm_predicted_pasal': majority_vote_pasal(svm_cases),
        'cosine_predicted_solution': reuse_solution(cos_cases),
        'svm_predicted_solution': reuse_solution(svm_cases),
        'cosine_top5_ids': ';'.join([r['case_id'] for r in cos_cases]),
        'svm_top5_ids': ';'.join([r['case_id'] for r in svm_cases]),
    })

pred_df = pd.DataFrame(predictions)
pred_df.to_csv(RESULTS_DIR / 'predictions.csv', index=False)
print('Predictions generated and saved to data/results/predictions.csv')

Predictions generated and saved to data/results/predictions.csv
